# Phase 3-4 — RAG 평가 파이프라인 (evaluator.py)

**목표:** RAG가 실제로 얼마나 잘 동작하는지 수치로 측정하는 평가 파이프라인을 구현한다.

**이 노트북을 마치면:**
- [ ] Hit Rate(결정론)를 집합 연산으로 계산할 수 있다
- [ ] key_facts 결정론 채점을 구현할 수 있다
- [ ] LLM-as-judge 프롬프트를 설계하고 점수를 파싱할 수 있다

**완성 후 연결:**
`src/evaluator.py`의 `compute_hit()`, `keyfact_score()`, `judge_answers()` 구현
```

## 섹션 1 — Hit Rate (결정론 지표)

Hit Rate = 정답이 있는 turn이 검색된 청크에 포함된 비율.

```
source_turns: [2, 5]          # QA 쌍에 명시된 정답 위치
included_turns: {1, 2, 4}    # 검색 청크의 turn 범위 집합

Hit Rate = |{2, 5} ∩ {1, 2, 4}| / |{2, 5}|
         = |{2}| / 2 = 0.5
```

**왜 결정론인가?** LLM 없이 집합 연산만으로 계산. 실행마다 동일한 값.
**baseline에서의 의미:** 고정 크기 청크가 정답 turn을 얼마나 포함했는가.

In [ ]:
# 집합 연산 복습
source = {2, 5}
included = {1, 2, 4}

intersection = source & included
print(f"교집합: {intersection}")
print(f"hit: {len(intersection)} / {len(source)} = {len(intersection)/len(source):.2f}")

### 미니 실습

In [ ]:
def compute_hit(source_turns: list[int], included_turns: set[int]) -> float:
    """source_turns 중 included_turns에 있는 비율 (0.0 ~ 1.0).
    
    source_turns가 비어 있으면 1.0 반환 (평가 불가 케이스 중립 처리).
    힌트: set 교집합 연산 활용
    """
    if not source_turns:
        return 1.0
    
    intersection = set(source_turns) & included_turns
    return len(intersection) / len(source_turns)

In [ ]:
assert compute_hit([2, 5], {1, 2, 4}) == 0.5,    "절반만 포함"
assert compute_hit([2, 5], {1, 2, 4, 5}) == 1.0,  "전부 포함"
assert compute_hit([2, 5], {1, 3, 4}) == 0.0,     "하나도 없음"
assert compute_hit([], {1, 2, 3}) == 1.0,          "source 없으면 1.0"
assert compute_hit([3], {3}) == 1.0,               "단일 턴 매칭"
print("✓ 통과")

### 본 실습

In [ ]:
def average_hit_rate(results: list[dict]) -> float:
    """여러 QA 결과의 평균 Hit Rate를 계산한다.
    
    results 각 항목: {"source_turns": list[int], "included_turns": set[int], ...}
    results가 비어 있으면 0.0 반환.
    
    힌트: compute_hit() 활용, sum(...) / len(results)
    """
    if not results:
        return 0.0
    
    hit = []
    for qa in results:
        hit.append(compute_hit(qa['source_turns'], qa['included_turns']))

    return sum(hit) / len(hit)


In [ ]:
results = [
    {"source_turns": [2, 5], "included_turns": {1, 2, 4}},   # hit=0.5
    {"source_turns": [1, 3], "included_turns": {1, 2, 3}},   # hit=1.0
    {"source_turns": [4],    "included_turns": {1, 2}},       # hit=0.0
]
avg = average_hit_rate(results)
assert abs(avg - 0.5) < 1e-6, f"(0.5+1.0+0.0)/3=0.5, 실제: {avg:.3f}"
assert average_hit_rate([]) == 0.0, "빈 목록 → 0.0"
print(f"평균 Hit Rate: {avg:.3f}")
print("✓ 통과")

**연결:** `compute_hit()`은 `evaluator.py run_baseline()` 안에서
각 QA 쌍의 `source_turns`와 검색된 청크의 turn 범위를 비교할 때 호출된다.
`average_hit_rate()`는 모드 전체 성능 요약 지표로 쓰인다.

---
## 섹션 2 — key_facts 결정론 채점

답변에 핵심 값이 들어있는가를 단순 문자열 포함으로 판단.
LLM 판단 없이 빠르고 재현 가능하다.

**정규화 (`_norm`):** 소문자 + 콤마/공백 제거
- "1,024" → "1024"
- "fastembed" → "fastembed"

단일 질문 → **Correctness** / 멀티홉 질문 → **Completeness** 역할.


In [ ]:
# 매칭용 정규화를 단계별로 확인
s = "1,024 MB"

step1 = s.lower()
step2 = step1.replace(",", "")
step3 = step2.replace(" ", "")

print(f"원본:      {s!r}")
print(f"소문자:    {step1!r}")
print(f"콤마 제거: {step2!r}")
print(f"공백 제거: {step3!r}")
# → "1024mb"  (소문자 + 콤마/공백 없음 = 매칭 기준)


### 미니 실습

In [ ]:
def _norm(s: str) -> str:
    """매칭용 정규화: 소문자 + 콤마/공백 제거.
    
    힌트: s.lower().replace(",", "").replace(" ", "")
    """
    return s.lower().replace(",", "").replace(" ", "")

In [ ]:
assert _norm("1,024") == "1024",         "콤마 제거"
assert _norm("Hello World") == "helloworld", "소문자 + 공백 제거"
assert _norm("0.3") == "0.3",            "변환 불필요"
assert _norm("fastembed") == "fastembed", "이미 정규화됨"
print("✓ 통과")

### 본 실습

In [ ]:
def keyfact_score(answer: str, key_facts: list[str]) -> float | None:
    """답변에 포함된 key_facts 비율 (0.0 ~ 1.0).
    
    key_facts가 비어 있으면 None 반환.
    각 fact가 _norm(answer)에 포함되는지 확인.
    
    힌트: _norm(fact) in _norm(answer)
    """
    if not key_facts:
        return None
    
    count = 0
    for fact in key_facts:
        if _norm(fact) in _norm(answer):
            count += 1
    return count / len(key_facts)

In [ ]:
assert keyfact_score("임계값은 0.3입니다", ["0.3"]) == 1.0
assert keyfact_score("35명이 참가했습니다", ["35", "20"]) == 0.5
assert keyfact_score("잘 모르겠습니다", ["0.3", "fastembed"]) == 0.0
assert keyfact_score("아무 답변", []) is None
assert keyfact_score("FastEmbed 설치 완료", ["fastembed"]) == 1.0
assert keyfact_score("1,024개 토큰", ["1024"]) == 1.0
print("✓ 통과")

**연결:** `keyfact_score()`는 `evaluator.py run_baseline()`에서
각 QA 답변의 `key_facts` 포함 여부를 확인할 때 호출된다.

---
## 섹션 3 — LLM-as-judge

key_facts로 잡지 못하는 서술형 답변 품질을 LLM이 0~10점으로 채점한다.

**프롬프트 설계 원칙:**
1. 채점 기준 명시
2. 기대 답변 제공
3. "숫자만 출력" 지시
4. max_tokens=10 (숫자 하나면 충분)

In [ ]:
example_prompt = """다음 질문에 대한 답변을 평가하세요.

질문: 이 대화에서 설정한 top_k 값은?
기대 답변: 3
실제 답변: top_k는 5로 설정했습니다.

평가 기준:
- 기대 답변의 핵심 정보가 포함됐는지 (0~10점)
- 사실과 다른 내용이 있으면 감점

숫자만 출력하세요 (예: 7)"""

print(example_prompt)
# 실제 답변이 틀렸으므로 → 0~2점 예상

### 미니 실습

In [ ]:
def build_judge_prompt(question: str, expected: str, answer: str) -> str:
    """LLM-as-judge 프롬프트를 생성한다.
    
    포함 요소:
    - 질문, 기대 답변, 실제 답변
    - 채점 기준 (핵심 정보 포함 여부, 사실 오류 감점)
    - "숫자만 출력" 지시
    """
    prompt = f"""다음 질문에 대한 답변을 평가하세요.

    질문: {question}
    기대 답변: {expected}
    실제 답변: {answer}

    평가 기준:
    - 기대 답변의 핵심 정보가 포함됐는지 (0~10점)
    - 사실과 다른 내용이 있으면 감점
    숫자만 출력하세요 (예: 7)
    """
    return prompt

In [ ]:
p = build_judge_prompt("임계값은?", "0.3", "임계값은 0.3입니다.")
assert "임계값" in p,            "질문 포함"
assert "0.3" in p,              "기대 답변 포함"
assert "임계값은 0.3입니다" in p, "실제 답변 포함"
assert "숫자" in p,             "'숫자만 출력' 지시 포함"
print("✓ 통과")
print("\n생성된 프롬프트:")
print(p)

### 미니 실습 2

In [ ]:
import re

def parse_judge_score(content: str) -> float:
    """LLM 출력에서 0~10 점수를 파싱한다.
    
    파싱 실패 시 0.0 반환.
    0~10 범위를 벗어나면 클리핑.
    
    힌트:
    - content.strip().split()[0].rstrip(".")
    - float() + try/except
    - max(0.0, min(10.0, score))
    """
    cleaned = content.strip().split()[0].rstrip(".")
    try:
        score = float(cleaned)
    except ValueError:
        return 0.0
    return max(0.0, min(10.0, score))

In [ ]:
assert parse_judge_score("7") == 7.0,    "정수"
assert parse_judge_score("7.5") == 7.5,  "소수"
assert parse_judge_score("7.") == 7.0,   "마침표 포함"
assert parse_judge_score("7 점") == 7.0, "숫자 뒤 텍스트"
assert parse_judge_score("abc") == 0.0,  "파싱 실패 → 0"
assert parse_judge_score("15") == 10.0,  "10 초과 → 클리핑"
assert parse_judge_score("-3") == 0.0,   "0 미만 → 클리핑"
print("✓ 통과")

### 본 실습: judge_single() 구현

`build_judge_prompt()` + `chat_completion()` + `parse_judge_score()`를 조합해
답변 하나를 0~10점으로 채점하는 함수를 완성한다.

In [ ]:
import sys
sys.path.insert(0, "../..")
from src._groq import chat_completion, load_env_key  # _groq.py 구현 완료 후 실행

def judge_single(question: str, expected: str, answer: str, api_key: str) -> float:
    """답변 하나를 0~10점으로 채점한다.
    
    힌트:
    - build_judge_prompt()로 프롬프트 생성
    - chat_completion([{"role": "user", "content": prompt}],
                      "llama-3.1-8b-instant", api_key, max_tokens=10, temperature=0.0)
    - parse_judge_score()로 점수 파싱
    - 예외 발생 시 0.0 반환
    """
    try:
        prompt = build_judge_prompt(question, expected, answer)
        content, _, _ = chat_completion([{'role': 'user', 'content' : prompt}],
                        "llama-3.1-8b-instant", api_key, max_tokens=10, temperature=0.0)
        
        return parse_judge_score(content)

    except Exception as e:
        print(f'오류: {e}')
        return 0.0


### 실제 채점 실험

`_groq.py` 구현 완료 후 실행.

In [ ]:
API_KEY = load_env_key()

test_cases = [
    {
        "question": "설치 명령어는?",
        "expected": "pip install fastembed",
        "answer":   "pip install fastembed 명령어로 설치합니다.",
    },
    {
        "question": "설치 명령어는?",
        "expected": "pip install fastembed",
        "answer":   "conda install 명령어를 쓰세요.",
    },
]

for tc in test_cases:
    score = judge_single(tc["question"], tc["expected"], tc["answer"], API_KEY)
    print(f"Q: {tc['question']}")
    print(f"A: {tc['answer']}")
    print(f"Score: {score}/10\n")

**연결:** `build_judge_prompt()`, `parse_judge_score()`, `judge_single()`은
`evaluator.py judge_answers()` 내부에서 각 결과 항목마다 호출된다.

---
## 스스로 정리해보기

노트북 완료 후 직접 작성:
- Hit Rate가 1.0이어도 Correctness가 낮을 수 있는 이유는?
    - LLM이 이미 알고 있는 지식으로 대답한 경우
- LLM-as-judge의 가장 큰 신뢰성 문제는 무엇이고, 이 프로젝트에서 어떻게 완화하는가?
    - 모르겠음.
        - 정답 : 가장 큰 문제는 채점 일관성이다. 같은 답변도 실행마다 점수가 달라질 수 있고, 길거나 자신감 있어 보이는 답변에 높은 점수를 주는 경향(길이 편향)이 있다.

        이 프로젝트에서 완화하는 방법:

        temperature=0.0 — 확률적 변동 제거
        max_tokens=10 — 숫자만 출력하도록 강제
        모드 비교 시 동일 채점자·동일 루브릭 유지
        절대 점수보다 모드 간 상대 비교에 집중하기 때문에, 채점자가 일관성만 유지하면 편향이 있어도 비교 결과는 유효하다.

- 고정 크기 청크 baseline에서 Hit Rate가 낮게 나오는 전형적인 패턴은?
    - 하나의 주제 중간에 청크가 나뉘는 경우